# Credit Risk Prediction Project


In [1]:
import pandas as pd
import numpy as np
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

## Step 1: Loading Data into SQLite

The raw dataset (`credit_risk_training.csv`, 150,000 records) is loaded into a SQLite database using pandas. The data is stored in a table named `borrowers`, which serves as the foundation for all subsequent SQL based exploration and cleaning.


In [2]:
df = pd.read_csv("credit_risk_training.csv")
conn = sqlite3.connect("credit_risk.db")
df.to_sql("borrowers", conn, if_exists="replace", index=False)
conn.close()

In [3]:
conn = sqlite3.connect("credit_risk.db")
query = "SELECT COUNT(*) AS total_rows FROM borrowers"
print(pd.read_sql(query, conn))

   total_rows
0      150000


## Step 2: Data Quality Checks and Exploratory Analysis

Before any modeling, the dataset is checked for missing values and anomalies using SQL queries.

Missing values are identified across all columns first. Then anomalies are checked: borrowers under 18 or over 100 should not appear in a real lending dataset, and the late payment columns contain values of 96 and 98 which are placeholder codes, not real counts.

A cleaned table, `borrowers_clean`, is created that removes underage and over-100 rows and converts the 96/98 placeholder values to NULL. Missing dependent counts are filled with 0 since no record likely means no dependents.


In [4]:
# Check all columns for missing values
df_check = pd.read_sql("SELECT * FROM borrowers", conn)

missing_results = {}
for col in df_check.columns:
    n_missing = df_check[col].isna().sum()
    if n_missing > 0:
        missing_results[col] = n_missing

if missing_results:
    print(pd.DataFrame.from_dict(missing_results, orient="index", columns=["missing_count"]))
else:
    print("No missing values found.")

                    missing_count
MonthlyIncome               29731
NumberOfDependents           3924


In [5]:
# Check for anomalous values
query_anomaly = """
SELECT
  SUM(CASE WHEN age <= 17 THEN 1 ELSE 0 END) AS age_underage,
  SUM(CASE WHEN age > 100 THEN 1 ELSE 0 END) AS age_over_100,
  SUM(CASE WHEN "NumberOfTime30-59DaysPastDueNotWorse" >= 96 THEN 1 ELSE 0 END) AS weird_30_59,
  SUM(CASE WHEN NumberOfTimes90DaysLate >= 96 THEN 1 ELSE 0 END) AS weird_90,
  SUM(CASE WHEN "NumberOfTime60-89DaysPastDueNotWorse" >= 96 THEN 1 ELSE 0 END) AS weird_60_89
FROM borrowers
"""
print(pd.read_sql(query_anomaly, conn))

   age_underage  age_over_100  weird_30_59  weird_90  weird_60_89
0             1            13          269       269          269


In [6]:
# Create clean table (drop first if it already exists)
conn.execute("DROP TABLE IF EXISTS borrowers_clean")
conn.commit()

create_clean = """
CREATE TABLE borrowers_clean AS
SELECT
  CustomerID,
  SeriousDlqin2yrs,
  RevolvingUtilizationOfUnsecuredLines,
  age,
  CASE WHEN "NumberOfTime30-59DaysPastDueNotWorse" >= 96 THEN NULL ELSE "NumberOfTime30-59DaysPastDueNotWorse" END AS "NumberOfTime30-59DaysPastDueNotWorse",
  DebtRatio,
  MonthlyIncome,
  NumberOfOpenCreditLinesAndLoans,
  CASE WHEN NumberOfTimes90DaysLate >= 96 THEN NULL ELSE NumberOfTimes90DaysLate END AS NumberOfTimes90DaysLate,
  NumberRealEstateLoansOrLines,
  CASE WHEN "NumberOfTime60-89DaysPastDueNotWorse" >= 96 THEN NULL ELSE "NumberOfTime60-89DaysPastDueNotWorse" END AS "NumberOfTime60-89DaysPastDueNotWorse",
  COALESCE(NumberOfDependents, 0) AS NumberOfDependents
FROM borrowers
WHERE age > 17 AND age <= 100
"""
conn.execute(create_clean)
conn.commit()
print("borrowers_clean created successfully")

borrowers_clean created successfully


In [7]:
# Default rate by age group
query_age = """
SELECT
  CASE
    WHEN age < 30 THEN 'under_30'
    WHEN age < 45 THEN '30_to_45'
    WHEN age < 60 THEN '45_to_60'
    ELSE 'over_60'
  END AS age_group,
  COUNT(*) AS total,
  ROUND(AVG(SeriousDlqin2yrs) * 100, 2) AS default_rate_percent
FROM borrowers_clean
GROUP BY age_group
ORDER BY default_rate_percent DESC
"""
print(pd.read_sql(query_age, conn))

  age_group  total  default_rate_percent
0  under_30   8820                 11.73
1  30_to_45  38982                  9.49
2  45_to_60  53879                  7.04
3   over_60  48305                  3.10


In [8]:
# Default rate by income group
query_income = """
SELECT
  CASE
    WHEN MonthlyIncome IS NULL THEN 'unknown'
    WHEN MonthlyIncome < 3000 THEN 'under_3000'
    WHEN MonthlyIncome < 6000 THEN '3000_to_6000'
    WHEN MonthlyIncome < 10000 THEN '6000_to_10000'
    ELSE 'over_10000'
  END AS income_group,
  COUNT(*) AS total,
  ROUND(AVG(SeriousDlqin2yrs) * 100, 2) AS default_rate_percent
FROM borrowers_clean
GROUP BY income_group
ORDER BY default_rate_percent DESC
"""
print(pd.read_sql(query_income, conn))

    income_group  total  default_rate_percent
0     under_3000  23322                  9.03
1   3000_to_6000  43812                  7.98
2  6000_to_10000  33343                  5.69
3        unknown  29724                  5.61
4     over_10000  19785                  4.33


## Step 3: Feature Engineering and Model Training

The cleaned table is loaded into a pandas dataframe for modeling.

Missing income values are handled with a flag approach: a new binary column `income_is_missing` marks which rows had no income data, and the income column itself is filled with 0. This way the model sees both the filled value and the information that it was originally missing.

The late payment NULLs (which came from the 96/98 placeholder codes removed in SQL) are filled with 0, since no real late payment record likely means no late payments occurred.

Three additional features are engineered:
- `total_past_due`: sum of all late payment counts across the three time windows.
- `has_severe_delinquency`: binary flag for whether the borrower was ever 90+ days late.
- `income_per_dependent`: monthly income divided by number of dependents plus one, capturing financial strain per person.

Two models are trained and compared: a Logistic Regression baseline and a Random Forest, both with balanced class weights to handle the imbalanced target (about 7% defaults).


In [9]:
# Load clean data into pandas
df = pd.read_sql("SELECT * FROM borrowers_clean", conn)
print(df.shape)
print(df.isna().sum())

(149986, 12)
CustomerID                                  0
SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse      269
DebtRatio                                   0
MonthlyIncome                           29724
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                   269
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse      269
NumberOfDependents                          0
dtype: int64


In [10]:
# Handle missing income with a flag instead of imputation
df["income_is_missing"] = df["MonthlyIncome"].isna().astype(int)
df["MonthlyIncome"] = df["MonthlyIncome"].fillna(0)

# Fill late payment NULLs with 0
df["NumberOfTime30-59DaysPastDueNotWorse"] = df["NumberOfTime30-59DaysPastDueNotWorse"].fillna(0)
df["NumberOfTimes90DaysLate"] = df["NumberOfTimes90DaysLate"].fillna(0)
df["NumberOfTime60-89DaysPastDueNotWorse"] = df["NumberOfTime60-89DaysPastDueNotWorse"].fillna(0)

# Fill missing dependents with 0
df["NumberOfDependents"] = df["NumberOfDependents"].fillna(0)

print(df.isna().sum())
print("income_is_missing value counts:")
print(df["income_is_missing"].value_counts())

CustomerID                              0
SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
income_is_missing                       0
dtype: int64
income_is_missing value counts:
income_is_missing
0    120262
1     29724
Name: count, dtype: int64


In [11]:
# Engineer new features
df["total_past_due"] = (
    df["NumberOfTime30-59DaysPastDueNotWorse"]
    + df["NumberOfTime60-89DaysPastDueNotWorse"]
    + df["NumberOfTimes90DaysLate"]
)
df["has_severe_delinquency"] = (df["NumberOfTimes90DaysLate"] > 0).astype(int)
df["income_per_dependent"] = df["MonthlyIncome"] / (df["NumberOfDependents"] + 1)

print(df[["total_past_due", "has_severe_delinquency", "income_per_dependent"]].describe())

       total_past_due  has_severe_delinquency  income_per_dependent
count   149986.000000           149986.000000          1.499860e+05
mean         0.400371                0.053798          3.615724e+03
std          1.101326                0.225620          8.155910e+03
min          0.000000                0.000000          0.000000e+00
25%          0.000000                0.000000          8.500000e+02
50%          0.000000                0.000000          2.597000e+03
75%          0.000000                0.000000          5.000000e+03
max         19.000000                1.000000          1.794060e+06


In [12]:
# Define features and target
feature_cols = [
    "RevolvingUtilizationOfUnsecuredLines", "age",
    "NumberOfTime30-59DaysPastDueNotWorse", "DebtRatio",
    "MonthlyIncome", "income_is_missing", "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate", "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse", "NumberOfDependents",
    "total_past_due", "has_severe_delinquency", "income_per_dependent"
]

X = df[feature_cols]
y = df["SeriousDlqin2yrs"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print("Default rate in train:", round(y_train.mean() * 100, 2), "%")
print("Default rate in test:", round(y_test.mean() * 100, 2), "%")

(112489, 14) (37497, 14)
Default rate in train: 6.68 %
Default rate in test: 6.68 %


In [13]:
# Baseline: Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg.fit(X_train_scaled, y_train)

log_reg_probs = log_reg.predict_proba(X_test_scaled)[:, 1]
log_reg_auc = roc_auc_score(y_test, log_reg_probs)

print("Logistic Regression AUC:", round(log_reg_auc, 4))

Logistic Regression AUC: 0.824


In [14]:
# Main model: Random Forest
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight="balanced",
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

rf_probs = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_probs)

print("Random Forest AUC:", round(rf_auc, 4))

Random Forest AUC: 0.8658


In [15]:
# Feature importances
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

RevolvingUtilizationOfUnsecuredLines    0.287076
total_past_due                          0.232841
NumberOfTime30-59DaysPastDueNotWorse    0.078076
NumberOfTimes90DaysLate                 0.074715
has_severe_delinquency                  0.073783
age                                     0.050176
NumberOfTime60-89DaysPastDueNotWorse    0.048740
DebtRatio                               0.042303
NumberOfOpenCreditLinesAndLoans         0.035968
MonthlyIncome                           0.023636
income_per_dependent                    0.023503
NumberRealEstateLoansOrLines            0.019614
NumberOfDependents                      0.007235
income_is_missing                       0.002333
dtype: float64


In [16]:
# Model evaluation
rf_preds = (rf_probs >= 0.5).astype(int)
print(classification_report(y_test, rf_preds))
print(confusion_matrix(y_test, rf_preds))

              precision    recall  f1-score   support

           0       0.98      0.84      0.90     34991
           1       0.24      0.73      0.36      2506

    accuracy                           0.83     37497
   macro avg       0.61      0.78      0.63     37497
weighted avg       0.93      0.83      0.86     37497

[[29229  5762]
 [  678  1828]]


In [17]:
# Export predictions for Power BI
results = X_test.copy()
results["actual_default"] = y_test.values
results["predicted_probability"] = rf_probs
results["predicted_default"] = rf_preds

results.to_csv("model_predictions.csv", index=False)
print(results.head())

        RevolvingUtilizationOfUnsecuredLines  age  \
15696                               0.698418   59   
54515                               0.828609   52   
56403                               0.004052   69   
32242                               0.034491   68   
114346                              0.706688   40   

        NumberOfTime30-59DaysPastDueNotWorse  DebtRatio  MonthlyIncome  \
15696                                    0.0   0.672467         2930.0   
54515                                    2.0   0.321229         9500.0   
56403                                    0.0   0.253357         5138.0   
32242                                    0.0   0.162419         2000.0   
114346                                   0.0   0.649668         4666.0   

        income_is_missing  NumberOfOpenCreditLinesAndLoans  \
15696                   0                                9   
54515                   0                               12   
56403                   0                         

## Step 4: Business Decision Analysis

A threshold sensitivity analysis is performed. For each possible decision cutoff between 0.05 and 0.95, the number of missed defaults (false negatives) and rejected good customers (false positives) is calculated, along with an estimated financial cost of missed defaults assuming an average loan size of 1,000 currency units.

This produces a trade off curve: a stricter cutoff catches more real defaults but also rejects more good customers. The output is exported for the second page of the Power BI dashboard, where a user can select a single threshold and see the resulting cost.


In [18]:
# Threshold sensitivity analysis
thresholds = np.arange(0.05, 1.0, 0.05)
avg_loan_amount = 1000  # business assumption: average loan size per customer

rows = []
for t in thresholds:
    preds_t = (rf_probs >= t).astype(int)
    tp = ((preds_t == 1) & (y_test == 1)).sum()
    fp = ((preds_t == 1) & (y_test == 0)).sum()
    fn = ((preds_t == 0) & (y_test == 1)).sum()
    tn = ((preds_t == 0) & (y_test == 0)).sum()
    rows.append({
        "threshold": round(t, 2),
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
        "estimated_loss_missed_defaults": fn * avg_loan_amount,
        "good_customers_rejected": fp
    })

threshold_df = pd.DataFrame(rows)
threshold_df.to_csv("threshold_analysis.csv", index=False)
print(threshold_df)

    threshold  true_positives  false_positives  false_negatives  \
0        0.05            2501            33675                5   
1        0.10            2484            28847               22   
2        0.15            2428            22109               78   
3        0.20            2345            17133              161   
4        0.25            2282            14334              224   
5        0.30            2229            12369              277   
6        0.35            2158            10697              348   
7        0.40            2087             9005              419   
8        0.45            1953             7332              553   
9        0.50            1828             5762              678   
10       0.55            1681             4460              825   
11       0.60            1532             3321              974   
12       0.65            1400             2523             1106   
13       0.70            1264             2037             124